<a href="https://colab.research.google.com/github/tousifo/ACM-Library/blob/master/MedMNIST_QNN_AllInOne.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pennylane medmnist scikit-learn matplotlib scikit-image
!pip install shap lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 119.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=9a86c454b81bf7eb0b3f0ba3b9fda1e60903b99a5029ee4e0a71db002ded3838
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a

In [ ]:
# =========================
# PREPROCESS
# =========================
import os, time, json, math, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, roc_auc_score

# Optional libs
try:
    import medmnist
    from medmnist import INFO
    print("✓ MedMNIST loaded")
except Exception as e:
    raise RuntimeError("Install medmnist first: pip install medmnist") from e

try:
    from PIL import Image
    HAVE_PIL = True
    print("✓ PIL loaded")
except:
    HAVE_PIL = False

# ---- Config
class Config:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    CACHE_DIR = Path("medmnist_cache")              # FIX: ensure exists + use as str
    BATCH_SIZE = 64
    EPOCHS_CORE = 80
    EPOCHS_VIA  = 40
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE = 20
    AUG_SEED = 42
    DO_CALIBRATION = False
    SNAPSHOT_ENSEMBLE = True
    SNAPSHOT_INTERVAL = 20
    CORE_DATASETS = ["BloodMNIST", "PneumoniaMNIST", "DermaMNIST"]
    VIABILITY_DATASET = "RetinaMNIST"

# FIX: create cache dir early
Config.CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"🔬 Device: {Config.DEVICE} | Cache: {Config.CACHE_DIR.resolve()}")

# ---- Small utils (used later)
def to_index(labels):
    if labels.dim() == 2 and labels.shape[1] == 1:
        return labels.squeeze(1).long()
    return labels.long()

def class_weights_from_counts(dist):
    total = sum(dist.values())
    return [total / (len(dist) * dist[i]) for i in sorted(dist.keys())]

# ---- Transforms & Datasets
def get_enhanced_transforms(n_channels, split='train'):
    mean_val = [.5]*n_channels
    std_val  = [.5]*n_channels
    if split == 'train':
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(20),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
            transforms.Normalize(mean=mean_val, std=std_val)
        ])
    else:
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=mean_val, std=std_val)
        ])

def get_datasets(name, cache_dir: Path):
    print(f"  📦 Loading {name} ...")
    info = INFO[name.lower()]
    n_channels = info['n_channels']
    n_classes = len(info['label'])
    DataClass = getattr(medmnist, info['python_class'])

    train_tf = get_enhanced_transforms(n_channels, 'train')
    test_tf  = get_enhanced_transforms(n_channels, 'test')

    # FIX: pass str(cache_dir) and ensure the path exists
    root = str(cache_dir)
    tr = DataClass(split='train', transform=train_tf,  download=True, root=root)
    va = DataClass(split='val',   transform=test_tf,   download=True, root=root)
    te = DataClass(split='test',  transform=test_tf,   download=True, root=root)

    # class distribution (train)
    train_labels = [int(tr[i][1].item() if hasattr(tr[i][1], 'item') else tr[i][1]) for i in range(len(tr))]
    dist = {"train": {i: train_labels.count(i) for i in range(n_classes)}}

    return tr, va, te, n_classes, n_channels, info, dist

def make_loaders(tr, va, te, batch_size=64, seed=42):
    def seed_worker(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed); random.seed(worker_seed)
    g = torch.Generator().manual_seed(seed)
    tr_loader = DataLoader(tr, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True,
                           worker_init_fn=seed_worker, generator=g)
    va_loader = DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    te_loader = DataLoader(te, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return tr_loader, va_loader, te_loader


✓ MedMNIST loaded
✓ PIL loaded
🔬 Device: cuda | Cache: /content/medmnist_cache


In [ ]:
# =========================
# MODEL + TRAIN/EVAL
# =========================
class QuantumCircuit(nn.Module):
    """Quantum Variational Circuit with Learnable Entanglement (simulated)."""
    def __init__(self, n_qubits, n_layers=4):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.theta = nn.Parameter(torch.randn(n_layers, n_qubits) * 0.1)
        self.phi   = nn.Parameter(torch.randn(n_layers, n_qubits) * 0.1)
        self.omega = nn.Parameter(torch.randn(n_layers, n_qubits) * 0.1)
        self.entangle_theta = nn.Parameter(torch.randn(n_layers, n_qubits-1) * 0.1)

    def forward(self, x):  # x: [B, n_qubits] in [-1,1]
        B = x.shape[0]
        state = torch.zeros(B, 2**self.n_qubits, dtype=torch.complex64, device=x.device)
        state[:, 0] = 1.0
        # encode
        for i in range(self.n_qubits):
            angle = x[:, i] * math.pi             # [B]
            phase = torch.exp(1j * angle.unsqueeze(-1) / 2)  # [B,1]
            state = state * phase
        # variational
        for L in range(self.n_layers):
            for i in range(self.n_qubits):
                angle = self.theta[L, i] + self.phi[L, i] * x[:, i]
                phase = torch.exp(1j * angle.unsqueeze(-1) / 2)
                state = state * phase
            for i in range(self.n_qubits - 1):
                s = self.entangle_theta[L, i]
                ent = (1 + 0.15 * torch.sin(s)).unsqueeze(0)  # [1] for broadcast
                state = state * ent
        # measurement (toy)
        probs = torch.abs(state) ** 2
        outs = []
        for i in range(self.n_qubits):
            outs.append(probs[:, : 2**(i+1)].sum(dim=1))
        return torch.stack(outs, dim=1)  # [B, n_qubits]

class LSTMFeatureExtractor(nn.Module):
    def __init__(self, input_channels=3, patch_size=4, hidden_dim=256, num_layers=3):
        super().__init__()
        self.patch_size = patch_size
        self.patch_embed = nn.Sequential(
            nn.Conv2d(input_channels, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU(),
        )
        self.seq_len = (28 // patch_size) ** 2
        self.patch_dim = 128 * patch_size * patch_size
        self.lstm = nn.LSTM(self.patch_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=0.3 if num_layers > 1 else 0, bidirectional=True)
        self.refine = nn.Sequential(
            nn.Linear(hidden_dim*2, hidden_dim*2), nn.LayerNorm(hidden_dim*2), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim*2, hidden_dim),   nn.LayerNorm(hidden_dim),   nn.GELU(), nn.Dropout(0.2),
        )

    def forward(self, x):  # x: [B,C,28,28]
        B = x.size(0)
        x = self.patch_embed(x)                  # [B,128,H,W]
        B,C,H,W = x.shape
        ps = self.patch_size
        x = x.unfold(2, ps, ps).unfold(3, ps, ps)      # [B,128,H/ps,W/ps,ps,ps]
        x = x.contiguous().view(B, C, -1, ps, ps)      # [B,128,N,ps,ps]
        x = x.permute(0,2,1,3,4).contiguous().view(B, -1, self.patch_dim)  # [B,N,patch_dim]
        lstm_out, _ = self.lstm(x)
        feat = self.refine(lstm_out[:, -1, :])
        return feat  # [B, hidden_dim]

class HybridLSTMQNN(nn.Module):
    def __init__(self, num_classes, input_channels=3, lstm_hidden=256, n_qubits=16, qnn_layers=4):
        super().__init__()
        self.lstm = LSTMFeatureExtractor(input_channels, patch_size=4, hidden_dim=lstm_hidden, num_layers=3)
        self.to_q = nn.Sequential(
            nn.Linear(lstm_hidden, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128),         nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, n_qubits), nn.Tanh()
        )
        self.qnn = QuantumCircuit(n_qubits, n_layers=qnn_layers)
        self.q_gate = nn.Sequential(
            nn.Linear(n_qubits, n_qubits*2), nn.LayerNorm(n_qubits*2), nn.GELU(),
            nn.Linear(n_qubits*2, n_qubits), nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.Linear(n_qubits + lstm_hidden, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 256),                    nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),                    nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
        )
        self.cls = nn.Linear(128, num_classes)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        if isinstance(m, (nn.LayerNorm, nn.BatchNorm2d)):
            nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        lstm_feat = self.lstm(x)
        q_in  = self.to_q(lstm_feat)
        q_out = self.qnn(q_in)
        gate  = self.q_gate(q_in)
        q_enh = q_out * gate + q_in * (1 - gate)
        fused = torch.cat([q_enh, lstm_feat], dim=1)
        z = self.fusion(fused)
        return self.cls(z)

# ----- Loss helpers
def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    ce = F.cross_entropy(logits, targets, reduction='none')
    pt = torch.exp(-ce)
    return (alpha * (1-pt)**gamma * ce).mean()

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam

def mixup_criterion(crit, pred, y_a, y_b, lam):
    return lam * crit(pred, y_a) + (1-lam) * crit(pred, y_b)

def label_smoothing_loss(logits, targets, smoothing=0.1):
    n = logits.size(-1)
    true = torch.full_like(logits, smoothing/(n-1))
    true.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)
    return torch.mean(torch.sum(-true * F.log_softmax(logits, dim=-1), dim=-1))

# ----- Train
def train_model(num_classes, n_channels, loaders, epochs, save_prefix, class_weights=None):
    tr_loader, va_loader, _ = loaders
    model = HybridLSTMQNN(num_classes, input_channels=n_channels).to(Config.DEVICE)

    weights = torch.tensor(class_weights, dtype=torch.float32, device=Config.DEVICE) if class_weights else None
    ce = nn.CrossEntropyLoss(weight=weights) if weights is not None else nn.CrossEntropyLoss()

    lstm_params    = list(model.lstm.parameters())
    quantum_params = list(model.qnn.parameters()) + list(model.to_q.parameters())
    other_params   = [p for p in model.parameters() if (id(p) not in {id(q) for q in lstm_params+quantum_params})]

    optim = torch.optim.AdamW([
        {'params': lstm_params, 'lr': Config.LEARNING_RATE},
        {'params': quantum_params, 'lr': Config.LEARNING_RATE * 0.5},
        {'params': other_params, 'lr': Config.LEARNING_RATE * 1.2}
    ], weight_decay=Config.WEIGHT_DECAY)

    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optim, T_0=15, T_mult=2, eta_min=1e-6)

    best_val, patience = 0.0, 0
    snapshots = []
    pbar = tqdm(range(1, epochs+1), unit="epoch", desc="Training")

    for ep in pbar:
        model.train(); tot_loss = 0.0
        for x, y in tr_loader:
            x, y = x.to(Config.DEVICE), to_index(y).to(Config.DEVICE)
            if np.random.rand() < 0.5:
                mx, ya, yb, lam = mixup_data(x, y, alpha=0.2)
                logits = model(mx)
                loss = mixup_criterion(ce, logits, ya, yb, lam)
            else:
                logits = model(x)
                r = np.random.rand()
                if r < 0.3:   loss = focal_loss(logits, y)
                elif r < 0.8: loss = label_smoothing_loss(logits, y, 0.1)
                else:         loss = ce(logits, y)
            optim.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step(); tot_loss += loss.item()
        sched.step()

        # val
        model.eval(); correct=0; total=0
        with torch.no_grad():
            for x,y in va_loader:
                x,y = x.to(Config.DEVICE), to_index(y).to(Config.DEVICE)
                pred = model(x).argmax(1)
                correct += (pred==y).sum().item(); total += y.size(0)
        val_acc = correct/total; pbar.set_postfix(val_acc=f"{val_acc:.3f}", best=f"{best_val:.3f}", loss=f"{tot_loss/len(tr_loader):.4f}")

        if val_acc > best_val:
            best_val = val_acc; patience = 0
            torch.save(model.state_dict(), f"{save_prefix}_best.pt")
        else:
            patience += 1

        if Config.SNAPSHOT_ENSEMBLE and ep % Config.SNAPSHOT_INTERVAL == 0:
            sp = f"{save_prefix}_snapshot_ep{ep:02d}.pt"
            torch.save(model.state_dict(), sp); snapshots.append(sp)

        if patience >= Config.PATIENCE:
            print(f"\n  Early stopping at epoch {ep}")
            break

    model.load_state_dict(torch.load(f"{save_prefix}_best.pt", map_location=Config.DEVICE))
    return model, snapshots

# ----- Evaluate
def evaluate_model(model, loaders, num_classes, use_tta=False):
    _, _, te_loader = loaders
    model.eval(); logits_all=[]; labels_all=[]
    with torch.no_grad():
        for x,y in te_loader:
            x,y = x.to(Config.DEVICE), to_index(y).to(Config.DEVICE)
            if use_tta:
                outs = [model(x), model(torch.flip(x,[3])), model(torch.flip(x,[2]))]
                logits = torch.stack(outs).mean(0)
            else:
                logits = model(x)
            logits_all.append(logits.cpu()); labels_all.append(y.cpu())
    logits = torch.cat(logits_all); labels = torch.cat(labels_all)
    probs = torch.softmax(logits, dim=1); preds = logits.argmax(1)
    acc = (preds==labels).float().mean().item()
    f1  = f1_score(labels.numpy(), preds.numpy(), average='macro')
    try:
        auroc = roc_auc_score(labels.numpy(),
                              probs[:,1].numpy() if num_classes==2 else probs.numpy(),
                              multi_class=None if num_classes==2 else 'ovr',
                              average='macro' if num_classes>2 else None)
    except Exception:
        auroc = 0.0
    return {'acc':acc, 'f1':f1, 'auroc':auroc, 'logits':logits, 'y':labels}


In [6]:
# =========================
# RESULTS & ACCURACY
# =========================
def run_one_dataset(name, root_dir: Path):
    ds_dir = root_dir / name; ds_dir.mkdir(parents=True, exist_ok=True)
    print("\n" + "="*80 + f"\n🔬 Dataset: {name}\n" + "="*80)

    tr, va, te, K, nC, info, dist = get_datasets(name, Config.CACHE_DIR)
    loaders = make_loaders(tr, va, te, Config.BATCH_SIZE, Config.AUG_SEED)
    cweights = class_weights_from_counts(dist["train"])

    print("\n  🎓 Training...")
    tag = ds_dir / "model"
    epochs = Config.EPOCHS_CORE if name in Config.CORE_DATASETS else Config.EPOCHS_VIA
    model, _ = train_model(K, nC, loaders, epochs, str(tag), cweights)

    print("\n  📊 Evaluating...")
    plain = evaluate_model(model, loaders, K, use_tta=False)
    tta   = evaluate_model(model, loaders, K, use_tta=True)

    print(f"  ✓ Acc: {plain['acc']:.3f} | F1: {plain['f1']:.3f} | AUROC: {plain['auroc']:.3f}")
    print(f"  ✓ Acc (TTA): {tta['acc']:.3f} | F1 (TTA): {tta['f1']:.3f} | AUROC (TTA): {tta['auroc']:.3f}")

    summary = {
        "dataset": name,
        "acc": plain["acc"], "f1": plain["f1"], "auroc": plain["auroc"],
        "acc_tta": tta["acc"], "f1_tta": tta["f1"], "auroc_tta": tta["auroc"],
        "n_params": sum(p.numel() for p in model.parameters()),
    }
    with open(ds_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    return summary

def main():
    print("\n" + "="*80 + "\nLSTM+QNN HYBRID ARCHITECTURE\n" + "="*80)
    print(f"\n🔬 Device: {Config.DEVICE}")
    print(f"📊 Datasets: {Config.CORE_DATASETS}")
    print(f"🎓 Epochs: {Config.EPOCHS_CORE}")
    print(f"📈 Learning Rate: {Config.LEARNING_RATE}\n")

    root = Path("results") / time.strftime("%Y%m%d_%H%M%S")
    root.mkdir(parents=True, exist_ok=True)

    all_summaries = []
    for ds in Config.CORE_DATASETS:
        all_summaries.append(run_one_dataset(ds, root))

    print("\n" + "="*80 + "\n📋 RESULTS SUMMARY\n" + "="*80)
    print(f"{'Dataset':<20} {'Acc':<8} {'F1':<8} {'AUROC':<8}")
    print("-"*60)
    for s in all_summaries:
        print(f"{s['dataset']:<20} {s['acc']:<8.3f} {s['f1']:<8.3f} {s['auroc']:<8.3f}")
    print("="*60)
    avg_acc = float(np.mean([s['acc'] for s in all_summaries])) if all_summaries else 0.0
    print(f"\n✅ Average Accuracy: {avg_acc:.3f}")
    print(f"📁 Results: {root}")

if __name__ == "__main__":
    main()



LSTM+QNN HYBRID ARCHITECTURE

🔬 Device: cuda
📊 Datasets: ['BloodMNIST', 'PneumoniaMNIST', 'DermaMNIST']
🎓 Epochs: 80
📈 Learning Rate: 0.0003


🔬 Dataset: BloodMNIST
  📦 Loading BloodMNIST ...


100%|██████████| 35.5M/35.5M [00:49<00:00, 713kB/s]



  🎓 Training...


Training:   0%|          | 0/80 [00:00<?, ?epoch/s]


  Early stopping at epoch 64

  📊 Evaluating...
  ✓ Acc: 0.937 | F1: 0.929 | AUROC: 0.996
  ✓ Acc (TTA): 0.942 | F1 (TTA): 0.934 | AUROC (TTA): 0.997

🔬 Dataset: PneumoniaMNIST
  📦 Loading PneumoniaMNIST ...


...


  Early stopping at epoch 56

  📊 Evaluating...
  ✓ Acc: 0.899 | F1: 0.890 | AUROC: 0.000
  ✓ Acc (TTA): 0.904 | F1 (TTA): 0.897 | AUROC (TTA): 0.000

🔬 Dataset: DermaMNIST
  📦 Loading DermaMNIST ...


100%|██████████| 19.7M/19.7M [00:03<00:00, 6.34MB/s]



  🎓 Training...


Training:   0%|          | 0/80 [00:00<?, ?epoch/s]


  Early stopping at epoch 59

  📊 Evaluating...
  ✓ Acc: 0.702 | F1: 0.357 | AUROC: 0.877
  ✓ Acc (TTA): 0.706 | F1 (TTA): 0.367 | AUROC (TTA): 0.883

📋 RESULTS SUMMARY
Dataset              Acc      F1       AUROC   
------------------------------------------------------------
BloodMNIST           0.937    0.929    0.996   
PneumoniaMNIST       0.899    0.890    0.000   
DermaMNIST           0.702    0.357    0.877   

✅ Average Accuracy: 0.846
📁 Results: results/20251012_160012
